# SutdyBot: un asistente para organizar tus estudios

In [48]:
import gradio as gr
import random
import spacy

nlp = spacy.load("es_core_news_sm")

In [49]:
# Reiniciamos el diccionario del perfil del usuario
perfil_usuario = {
    "nombre": None,
    "asignatura": None,
    "horas": None,
    "dificultad": None,
    "apuntes": None,
    "distracciones": None,
    "horario": None,
    "evaluacion": None,
    "estres": None,
    "meta": None,
}

# Flujo estricto de 10 estados para la evaluación
ESTADOS = [
    "PEDIR_NOMBRE",
    "PEDIR_ASIGNATURA",
    "PEDIR_HORAS",
    "PEDIR_DIFICULTAD",
    "PEDIR_APUNTES",
    "PEDIR_DISTRACCIONES",
    "PEDIR_HORARIO",
    "PEDIR_EVALUACION",
    "PEDIR_ESTRES",
    "PEDIR_META",
    "FIN",
]

indice_estado = 0


def extraer_nombre(texto):
    """Usa SpaCy para identificar el nombre propio en frases como 'Me llamo Adán'."""
    doc = nlp(texto)
    nombres_propios = [
        token.text
        for token in doc
        if token.pos_ == "PROPN" and token.text.lower() not in ["llamo", "nombre"]
    ]

    if nombres_propios:
        return nombres_propios[0].capitalize()
    return texto.capitalize()


def generar_plan_estudio():
    """Genera una conclusión detallada con consejos dinámicos según las respuestas."""
    p = perfil_usuario

    consejo_tiempo = ""
    try:
        horas_num = int("".join(filter(str.isdigit, p["horas"])))
        if horas_num < 2:
            consejo_tiempo = "Dedicas menos de 2 horas al día. Te sugiero aumentar al menos 30 minutos diarios usando bloques libres para no saturarte."
        else:
            consejo_tiempo = "¡Buen tiempo de estudio! Asegúrate de incluir descansos de 5 minutos por cada 25 de enfoque."
    except:
        consejo_tiempo = (
            f"Mantén la constancia en esas {p['horas']} que dedicas habitualmente."
        )

    consejo_distraccion = ""
    dist_low = p["distracciones"].lower()
    if "celular" in dist_low or "teléfono" in dist_low or "redes" in dist_low:
        consejo_distraccion = "Para mitigar el uso de pantallas, utiliza apps de bloqueo como Forest o deja el teléfono en otra habitación."
    else:
        consejo_distraccion = f"Crea un espacio libre de interrupciones enfocado en reducir el impacto de: {p['distracciones']}."

    conclusion = (
        f"📋 PLAN DE ESTUDIO PERSONALIZADO PARA {p['nombre'].upper()}\n"
        f"--------------------------------------------------\n"
        f"• Asignatura: {p['asignatura']}\n"
        f"• Dificultad principal: {p['dificultad']}\n"
        f"• Mayor distracción: {p['distracciones']}\n"
        f"• Horario ideal: De {p['horario']}\n"
        f"• Meta: {p['meta']}\n\n"
        f"💡 CONCLUSIONES Y CONSEJOS DE MEJORA:\n"
        f"1. Sobre el Tiempo: {consejo_tiempo}\n"
        f"2. Sobre los Distractores: {consejo_distraccion}\n"
        f"3. Estrategia de Examen: Como tu dificultad es '{p['dificultad']}' y tienes una evaluación ({p['evaluacion']}), "
        f"te recomiendo priorizar la práctica activa (hacer ejercicios o esquemas) en tus horarios de mayor energía ({p['horario']}).\n\n"
        f"¡Mucho éxito! Has completado tu diagnóstico de 10 pasos de forma excelente."
    )
    return conclusion


def responder_chat(mensaje, historial):
    """Función que conecta el backend del bot con la interfaz gráfica de Gradio."""
    global indice_estado

    if indice_estado >= len(ESTADOS) or ESTADOS[indice_estado] == "FIN":
        return "El diagnóstico ha finalizado con éxito. Reinicia la celda para volver a empezar."

    estado_actual = ESTADOS[indice_estado]
    respuesta_bot = ""

    # --- AQUÍ ESTÁ EL CAMBIO EN "PEDIR_NOMBRE" ---
    if estado_actual == "PEDIR_NOMBRE":
        saludos = ["hola", "buen", "tarde", "noche", "que tal", "saludos"]

        # Si el usuario solo saluda (ej: "Hola!"), el bot responde pero NO avanza de estado
        if any(s in mensaje.lower() for s in saludos):
            respuesta_bot = "¡Hola! Qué bueno tenerte aquí. Para comenzar el diagnóstico, ¿cómo te llamas?"
        else:
            # Si introduce su nombre, lo extrae con SpaCy y avanza al siguiente estado
            perfil_usuario["nombre"] = extraer_nombre(mensaje)
            indice_estado += 1
            respuesta_bot = f"Mucho gusto, {perfil_usuario['nombre']}. ¿Qué asignatura o materia es tu mayor desafío actualmente?"

    elif estado_actual == "PEDIR_ASIGNATURA":
        perfil_usuario["asignatura"] = mensaje
        indice_estado += 1
        respuesta_bot = f"Entendido. ¿Cuántas horas al día le dedicas formalmente a estudiar esa materia?"

    elif estado_actual == "PEDIR_HORAS":
        perfil_usuario["horas"] = mensaje
        indice_estado += 1
        respuesta_bot = "¿Qué es lo que más te cuesta de esa materia? (Ej: memorizar, hacer ejercicios, entender la teoría)"

    elif estado_actual == "PEDIR_DIFICULTAD":
        perfil_usuario["dificultad"] = mensaje
        indice_estado += 1
        respuesta_bot = "Cuando estás en clases, ¿tomas apuntes a mano, en computador o prefieres solo escuchar?"

    elif estado_actual == "PEDIR_APUNTES":
        perfil_usuario["apuntes"] = mensaje
        indice_estado += 1
        respuesta_bot = "A la hora de estudiar en casa, ¿cuál es tu mayor distracción? (Ej: el celular, redes sociales, el ruido)"

    elif estado_actual == "PEDIR_DISTRACCIONES":
        perfil_usuario["distracciones"] = mensaje
        indice_estado += 1
        respuesta_bot = "¿En qué momento del día sientes que tu cerebro funciona mejor? (mañana, tarde o noche)"

    elif estado_actual == "PEDIR_HORARIO":
        perfil_usuario["horario"] = mensaje
        indice_estado += 1
        respuesta_bot = "¿Tienes alguna evaluación o examen importante programado para las próximas semanas? (sí/no / cuándo)"

    elif estado_actual == "PEDIR_EVALUACION":
        perfil_usuario["evaluacion"] = mensaje
        indice_estado += 1
        respuesta_bot = "Del 1 al 5, ¿qué tan estresado(a) te sientes con respecto a tus estudios actualmente?"

    elif estado_actual == "PEDIR_ESTRES":
        perfil_usuario["estres"] = mensaje
        indice_estado += 1
        respuesta_bot = "Por último, ¿cuál es tu meta de calificación o nota para este periodo? (Ej: aprobar, nota máxima)"

    elif estado_actual == "PEDIR_META":
        perfil_usuario["meta"] = mensaje
        indice_estado = len(ESTADOS) - 1
        respuesta_bot = generar_plan_estudio()

    return respuesta_bot

In [ ]:
# Reiniciamos el índice de estado antes de lanzar la app
indice_estado = 0

# Creamos la interfaz manteniendo solo lo esencial que no cambia entre versiones
demo = gr.ChatInterface(
    fn=responder_chat,
    title="🤖 StudyBot 2.0 - Tu Asesor Académico",
    description=(
        "¡Bienvenido! Para iniciar tu diagnóstico académico de 10 pasos, "
        "simplemente saluda en el cuadro inferior y presiona Enter."
    ),
    textbox=gr.Textbox(
        placeholder="Escribe aquí tu respuesta...", container=False, scale=7
    ),
)

# Lanzamos la aplicación dentro del Notebook
demo.launch(inline=True, share=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\10. Taller de aplicación de IA\venv-10\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\10. Taller de aplicación de IA\venv-10\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\10. Taller de aplicación de IA\venv-10\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\adanm\Documentos\Magister